In [14]:
import numpy as np
import tensorflow as tf
from spektral.data import Dataset, DisjointLoader, Graph
from spektral.layers import GINConv, GlobalAvgPool
import scipy.sparse as sp
import os
from tensorflow.keras.metrics import categorical_accuracy
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import f1_score

from utils import *

from tensorflow.python.ops.numpy_ops import np_config
np_config.enable_numpy_behavior()
import warnings
warnings.filterwarnings("ignore", category=Warning)

# Load CFG json files as spektral graph objects

In [2]:
################################################################################
# Load data
################################################################################ 
class GraphData(Dataset):
    

    def __init__(self, cfg_path, **kwargs):
        self.cfg_path = cfg_path

        super().__init__(**kwargs)

    def read(self):
        
        file_list = os.listdir(self.cfg_path)
        file_list_x_y = list(filter(lambda x: '_sparse_matrix' not in x and '.npz' in x, file_list))
        
        #print(len(file_list_x_y))
        output = []
        
        
       
        for filepath in file_list_x_y:

            #full path of node attribute and label
            fullpath = os.path.join(self.cfg_path, filepath)
            #file path of adj matrix
            filepath_sp = filepath.split('.')[0] + "_sparse_matrix.npz"
            #full path pf adj matrix
            fullpath_sp = os.path.join(self.cfg_path, filepath_sp)
            #with open(fullpath_sp, 'rb') as f1:
            sparse_matrix = sp.load_npz(fullpath_sp)
            sparse_matrix = sparse_matrix.astype('float32')
            
            #If the sparse_matrix size is over 46000 by 46000 ,we skipped it since
            #we are unable to allocate that much memory for an array with that shape 
            if sparse_matrix.shape[0] > 46000: 
                continue
           
            #with open(fullpath, 'rb') as f2:
            data = np.load(fullpath)
            
            # Remove diagonal elements
            adj = sparse_matrix - sp.dia_matrix((sparse_matrix.diagonal()[np.newaxis, :], [0]), shape=sparse_matrix.shape)
            adj.eliminate_zeros()
            # Check that diag is zero:
            assert np.diag(adj.todense()).sum() == 0

            adj_triu = sp.triu(adj)
            adj_tuple = sparse_to_tuple(adj_triu)
            edges = adj_tuple[0]
        
                
   
            
            #important! filter out noisy data where the number of basic block is less than 10 and the number of non-self edges in the upper triangle is less than 3
            # if the graph is too large, we will run into oom problems
            if data["x"].shape[0] >=10 and edges.shape[0] >= 3 and sparse_matrix.shape[0] <=46000:
                output.append(Graph(x=data['x'], a= sparse_matrix, y=data['y']))
                
               
          
          

        return output




# Load malware data

In [3]:
path =  'data/graph_features/big-15/cfg_embeddings/cfg_embeddings'
malware = GraphData(path) 
malware = convert_label_integer(malware)

In [4]:
label = obtain_labels(malware)
uniques, counts = np.unique(label, return_counts=True)

percentages = dict(zip(uniques, counts * 100 / len(label)))

print(percentages)


# 1 Ramnit 	1541 	Worm 1
# 2 Lollipop 	2478 	Adware 
# 3 Kelihos_ver3 	2942 	Backdoor
# 4 Vundo 	475 	Trojan
# 5 Simda 	42 	Backdoor
# 6 Tracur 	751 	Trojan Downloader
# 7 Kelihos_ver1 	398 	Backdoor
# 8 Obfuscator.ACY 	1228 	Obfuscated malware
# 9 Gatak 	1013 	Backdoor
# 0 Normal 	6392 	

{0: 14.032303239660163, 1: 23.116422369526656, 2: 27.252357389599478, 3: 3.995892073569228, 4: 0.3921202502100644, 5: 6.862104378676127, 6: 3.5850994304920176, 7: 11.31547007749043, 8: 9.448230790775838}


# Load benign data source

In [5]:
path_1 =  'data/graph_features/benign_source/dataset1/cfg_embeddings'
normal_1_source = GraphData(path_1)
normal_1_source = convert_label_integer(normal_1_source)

path_2 =  'data/graph_features/benign_source/dataset2/cfg_embeddings'
normal_2_source = GraphData(path_2)
normal_2_source = convert_label_integer(normal_2_source)

path_3 =  'data/graph_features/benign_source/dataset3/cfg_embeddings'
normal_3_source = GraphData(path_3)
normal_3_source = convert_label_integer(normal_3_source)

path_4 =  'data/graph_features/benign_source/dataset4/cfg_embeddings'
normal_4_source = GraphData(path_4)
normal_4_source = convert_label_integer(normal_4_source)

source_normal = merge_dataset(normal_1_source, normal_2_source)
source_normal = merge_dataset(source_normal, normal_3_source)
source_normal = merge_dataset(source_normal, normal_4_source)
print(len(source_normal))

6510


# Load benign data target

In [6]:
path_1 =  'data/graph_features/benign_target/dataset1/cfg_embeddings'
normal_1_target = GraphData(path_1)
normal_1_target = convert_label_integer(normal_1_target)

path_2 =  'data/graph_features/benign_target/dataset2/cfg_embeddings'
normal_2_target = GraphData(path_2)
normal_2_target = convert_label_integer(normal_2_target)

path_3 =  'data/graph_features/benign_target/dataset3/cfg_embeddings'
normal_3_target = GraphData(path_3)
normal_3_target = convert_label_integer(normal_3_target)

path_4 =  'data/graph_features/benign_target/dataset4/cfg_embeddings'
normal_4_target = GraphData(path_4)
normal_4_target = convert_label_integer(normal_4_target)

target_normal = merge_dataset(normal_1_target, normal_2_target)
target_normal = merge_dataset(target_normal , normal_3_target)
target_normal = merge_dataset(target_normal , normal_4_target)
print(len(target_normal))

5768


# Build our DA model

In [7]:

################################################################################
# Build model
################################################################################
class GIN0(Model):
    def __init__(self, channels, n_layers):
        super().__init__()
        self.conv1 = GINConv(channels, epsilon=0, mlp_hidden=[channels, channels])
        self.convs = []
        for _ in range(1, n_layers):
            self.convs.append(
                GINConv(channels, epsilon=0, mlp_hidden=[channels, channels])
            )
        self.pool = GlobalAvgPool()
        self.dense1 = Dense(channels, activation="relu")
      
        

    def call(self, inputs):
        x, a, i = inputs
        x = self.conv1([x, a])
        for conv in self.convs:
            x = conv([x, a])
        x = self.pool([x, i])
        x = self.dense1(x)
      
        return x

    
    

In [8]:
class DANN_GIN(object):
    def __init__(self,loader_source_train, 
                 loader_target_train,
                 loader_target_test, GIN, n_classes,
                 epochs=90):

        #source train and test dataset
        self.loader_source_tr = loader_source_train
        #self.loader_source_te= loader_source_test
        
        
        # Target train and test dataset
        
        self.loader_target_tr = loader_target_train
        self.loader_target_te= loader_target_test


        self.n_classes = n_classes
    
        
        #Latent dim for AE/VAE
        self.latent_dim = 256 #25
        
        
        self.generator = GIN 
        self.epochs = epochs 
        
       
        #Classifier
        
        class_input = Input(shape=(256,))
        #x = latent(class_input)
        x= Dense(256, activation = "relu")(class_input)
        class_output = Dense(self.n_classes, activation = "softmax")(x)
        
        self.classifier = Model(class_input, class_output, name="classifier")
        
        #Discriminator
        
        disc_input = Input(shape=(256,))
        #x = latent(disc_input)
        x = Dense(256, activation = "relu")(disc_input)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        x = Dense(256, activation = "relu")(x)
        x = BatchNormalization()(x)
        x = Activation('relu')(x)
        disc_output = Dense(2, activation = "softmax")(x)
        
        self.discriminator = Model(disc_input, disc_output, name="discriminator")

      
        
      
        self.loss = tf.keras.losses.CategoricalCrossentropy()

        
        self.lr = 0.001 
        self.momentum = 0.9
        self.alpha = 0.0002

        
        
        
        self.task_optimizer=  Adam(1e-3)
        self.gen_optimizer = Adam(1e-3)
        self.disc_optimizer = Adam(1e-3)
        
        self.train_task_loss = tf.keras.metrics.Mean()
        self.train_disc_loss = tf.keras.metrics.Mean()
        self.train_gen_loss = tf.keras.metrics.Mean()
        self.train_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        self.train_target_task_accuracy = tf.keras.metrics.CategoricalAccuracy()

   
        
        self.test_target_task_loss = tf.keras.metrics.Mean()
        self.test_target_task_accuracy = tf.keras.metrics.CategoricalAccuracy()
        



        
        self.batch_size = 16

        


    def train_batch(self, x_source_train, y_source_train, x_target_train, y_target_train, epoch):
        
        
        source = np.tile([1,0], (y_source_train.shape[0], 1))
        target = np.tile([0,1], (y_target_train.shape[0], 1))
        
        target_fake = np.tile([0,1], (y_source_train.shape[0], 1))
        source_fake = np.tile([1,0], (y_target_train.shape[0], 1))
        
        with tf.GradientTape() as disc_tape:
            y_domain_pred_source = self.discriminator(self.generator(x_source_train, training=True), training=True)
            y_domain_pred_target = self.discriminator(self.generator(x_target_train, training=True), training=True)
            
            disc_loss = self.loss(source, y_domain_pred_source) +  self.loss(target, y_domain_pred_target)  
            
        disc_grad = disc_tape.gradient(disc_loss, self.discriminator.trainable_variables)  
        self.disc_optimizer.apply_gradients(zip(disc_grad, self.discriminator.trainable_variables))
        self.train_disc_loss(disc_loss)

        with tf.GradientTape() as task_tape, tf.GradientTape() as gen_tape:
            
            #Forward pass
            y_class_pred_source = self.classifier(self.generator(x_source_train, training=True), training=True)
            y_class_pred_target = self.classifier(self.generator(x_target_train, training=True), training=True)
            y_domain_pred_source = self.discriminator(self.generator(x_source_train, training=True), training=True)
            y_domain_pred_target = self.discriminator(self.generator(x_target_train, training=True), training=True)
            
            
            task_loss = self.loss(y_target_train, y_class_pred_target) + 0.1*self.loss(y_source_train, y_class_pred_source)  
            adv_loss = self.loss(target_fake, y_domain_pred_source) +  self.loss(source_fake, y_domain_pred_target)   
            gen_loss = task_loss +  adv_loss*0.1
            
        
        
        # Compute gradients   
        task_grad = task_tape.gradient(task_loss, self.classifier.trainable_variables)
        gen_grad = gen_tape.gradient(gen_loss, self.generator.trainable_variables)
       
        
        # Update weights 
        self.task_optimizer.apply_gradients(zip(task_grad, self.classifier.trainable_variables))
        self.gen_optimizer.apply_gradients(zip(gen_grad, self.generator.trainable_variables)) 
        
            

        self.train_task_loss(task_loss)
        self.train_task_accuracy(y_source_train, y_class_pred_source)
        self.train_target_task_accuracy(y_target_train, y_class_pred_target)
        self.train_gen_loss(gen_loss)
            
            


        return
    
    
    def test_batch(self, x_target_test, y_target_test):
       
        # y_class_pred = self.classifier(self.generator(x_source_test, training=False), training=False)
        y_target_class_pred = self.classifier(self.generator(x_target_test, training=False), training=False)
        
            
        #self.test_task_loss.update_state(y_source_test, y_class_pred)
        self.test_target_task_loss(y_target_test, y_target_class_pred)
        #self.test_task_accuracy.update_state(y_source_test, y_class_pred)
        self.test_target_task_accuracy(y_target_test, y_target_class_pred)
        #self.test_target_f1_score.update_state(y_target_test, y_target_class_pred)
        
       
        
        return 
    
    def evaluate(self, loader):
        output = []
        loss_fn = tf.keras.losses.CategoricalCrossentropy()
        step = 0
        while step < loader.steps_per_epoch:
            step += 1
            inputs, target = loader.__next__()
            pred = self.classifier(self.generator(inputs, training=False), training=False)
            outs = (
                loss_fn(target, pred),
                tf.reduce_mean(categorical_accuracy(target, pred)),
                len(target),  # Keep track of batch size
            )          
            output.append(outs)
            if step == loader.steps_per_epoch:
                output = np.array(output)
                return np.average(output[:, :-1], 0, weights=output[:, -1])

      
            


    
    def log_train(self):
        
        
        log_format = 'C_loss train: {:.4f}, Acc train source: {:.2f} , Acc train target: {:.2f}\n'+'D_loss train: {:.4f}, G_loss train: {:.4f}'

        message = log_format.format(
                 self.train_task_loss.result(),
                 self.train_task_accuracy.result()*100,
                 self.train_target_task_accuracy.result()*100,
                 self.train_disc_loss.result(),
                 self.train_gen_loss.result())
        

        self.reset_metrics('train')
        


        return message 
    
    def log_test(self):
        
        
        log_format = "C_loss test target: {:.4f}, Acc test target: {:.2f}"

        message = log_format.format(
                 #self.test_task_loss.result(),
                 #self.test_task_accuracy.result()*100,
                 self.test_target_task_loss.result(),
                 self.test_target_task_accuracy.result()*100)
                 #self.test_target_f1_score.result()*100)
        

        self.reset_metrics('test')


        return message 
    
    def reset_metrics(self, target):

        if target == 'train':
            self.train_task_loss.reset_states()
            self.train_task_accuracy.reset_states()
            self.train_disc_loss.reset_states()
            self.train_gen_loss.reset_states()
            
        
        
        if target == 'test':
            self.test_target_task_loss.reset_states()
            self.test_target_task_accuracy.reset_states()
        

        return 
    
    def train(self):
        epoch = step = 0
    

        for (source_batch, source_labels), (target_batch, target_labels) in zip(self.loader_source_tr, self.loader_target_tr):
            step +=1 
            self.train_batch(source_batch, source_labels, target_batch, target_labels, epoch)
            if step == min(self.loader_source_tr.steps_per_epoch, self.loader_target_tr.steps_per_epoch):
                step = 0
                epoch +=1  
                if epoch % 10 ==0:
                    print('Epoch: {}'.format(epoch))
                    print(self.log_train())                 
                    results_te = self.evaluate(self.loader_target_te)
                    print("Test results - Loss: {:.3f} - Acc: {:.3f}".format(*results_te))
                    
                    

        return self.generator, self.classifier
                


        
                
            
            

# Training 

## Set ramnit as the target domain

In [9]:
# ramnit label is 0
target_malware, source_malware = filter_label(malware, [0])


# convert it to binary labels
target_malware = binary_label(target_malware, True)
source_malware  = binary_label(source_malware, True)
target_normal = binary_label(target_normal, False)
source_normal = binary_label(source_normal, False)

print("Target malware size: {}".format(len(target_malware)))
print("Target normal size: {}".format(len(target_normal)))
print("Source malware size: {}".format(len(source_malware)))
print("Source normal size: {}".format(len(source_normal)))
           
           

Target malware size: 1503
Target normal size: 5768
Source malware size: 9208
Source normal size: 6510


In [10]:
# Split the  malware/benign dataset into train and test set
target_malware_train, target_malware_test = train_test_split(target_malware, 0.5)
print(len(target_malware_train))
print(len(target_malware_test))

target_normal_train, target_normal_test = train_test_split(target_normal, 0.5)
print(len(target_normal_train))
print(len(target_normal_test))

751
752
2884
2884


In [11]:
#combine the normal and malware dataset
source = merge_dataset(source_normal, source_malware)

#source dataset train and test split
source_train, source_test = train_test_split(source, 0.75)

# we do the same for the target dataset
target_train = merge_dataset(target_normal_train, target_malware_train)
target_test = merge_dataset(target_normal_test, target_malware_test)



print("Target train dataset size: {}".format(len(target_train)))
print("Target test dataset size: {}".format(len(target_test)))
print("Source train dataset size: {}".format(len(source_train)))
print("Source test dataset size: {}".format((len(source_test))))

Target train dataset size: 3635
Target test dataset size: 3636
Source train dataset size: 11788
Source test dataset size: 3930


In [15]:
samples = [20, 50, 100, 200, 300, 500]

channels = 256  # Hidden units
layers = 3  # GIN layers
# We have limited the number of training epochs to 20 to minimize training time.
# You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
# However, for better model performance, consider increasing the number of epochs to those specified in the referenced paper
epochs = 20  
batch_size = 16  # Batch size
n_out = target_train.n_labels

for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))

    target_train_select, target_train_re = subsample(target_train, size)

    loader_source_tr = DisjointLoader(source_train, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_source_te = DisjointLoader(source_test,  batch_size=batch_size)


    loader_target_tr = DisjointLoader(target_train_select, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_target_te = DisjointLoader(target_test, batch_size=batch_size)

    # build model
    GIN = GIN0(channels, layers)


    model =DANN_GIN(loader_source_tr, loader_target_tr, loader_target_te, GIN, n_out, epochs==20) # change the epochs here as well


    G, C = model.train()


    ################################################################################
    # Evaluate model
    ################################################################################
    results = []
    step = 0

    while step < loader_target_te.steps_per_epoch:      
        step += 1
        inputs, target = loader_target_te.__next__()
        pred = C(G(inputs, training=False), training=False)
        results.append(
            (
               f1_score(np.argmax(target, axis=1), np.argmax(pred, axis=1), average='weighted')
            )
        )
    print("Done. Test f1: {}".format(np.mean(results, 0)))

        
       

--------------------------Sample size 20-------------------------
Epoch: 10
C_loss train: 0.1951, Acc train source: 80.62 , Acc train target: 96.50
D_loss train: 1.6379, G_loss train: 0.3660
Test results - Loss: 0.060 - Acc: 0.988
Epoch: 20
C_loss train: 0.1119, Acc train source: 89.69 , Acc train target: 97.50
D_loss train: 1.4851, G_loss train: 0.2779
Test results - Loss: 0.052 - Acc: 0.985
Done. Test f1: 0.9868003935151092
--------------------------Sample size 50-------------------------
Epoch: 10
C_loss train: 0.2497, Acc train source: 92.19 , Acc train target: 94.60
D_loss train: 1.5128, G_loss train: 0.4270
Test results - Loss: 0.388 - Acc: 0.816
Epoch: 20
C_loss train: 0.2247, Acc train source: 89.84 , Acc train target: 95.30
D_loss train: 1.3227, G_loss train: 0.4084
Test results - Loss: 0.026 - Acc: 0.992
Done. Test f1: 0.992251035394493
--------------------------Sample size 100-------------------------
Epoch: 10
C_loss train: 0.1785, Acc train source: 91.96 , Acc train target

## Set lollipop as the target domain

In [16]:
# lollipop label is 1
target_malware, source_malware = filter_label(malware, [1])


# convert it to binary labels
target_malware = binary_label(target_malware, True)
source_malware  = binary_label(source_malware, True)
target_normal = binary_label(target_normal, False)
source_normal = binary_label(source_normal, False)

print("Target malware size: {}".format(len(target_malware)))
print("Target normal size: {}".format(len(target_normal)))
print("Source malware size: {}".format(len(source_malware)))
print("Source normal size: {}".format(len(source_normal)))
           
           

Target malware size: 2476
Target normal size: 5768
Source malware size: 8235
Source normal size: 6510


In [17]:
# Split the  malware/benign dataset into train and test set
target_malware_train, target_malware_test = train_test_split(target_malware, 0.5)
print(len(target_malware_train))
print(len(target_malware_test))

target_normal_train, target_normal_test = train_test_split(target_normal, 0.5)
print(len(target_normal_train))
print(len(target_normal_test))

1238
1238
2884
2884


In [18]:
# combine the normal and malware dataset
source = merge_dataset(source_normal, source_malware)

# source dataset train and test split
source_train, source_test = train_test_split(source, 0.75)
# we do the same for the target dataset
target_train = merge_dataset(target_normal_train, target_malware_train)
target_test = merge_dataset(target_normal_test, target_malware_test)



print("Target train dataset size: {}".format(len(target_train)))
print("Target test dataset size: {}".format(len(target_test)))
print("Source train dataset size: {}".format(len(source_train)))
print("Source test dataset size: {}".format((len(source_test))))

Target train dataset size: 4122
Target test dataset size: 4122
Source train dataset size: 11058
Source test dataset size: 3687


In [19]:
samples = [20, 50, 100, 200, 300, 500]

channels = 256  # Hidden units
layers = 3  # GIN layers
# We have limited the number of training epochs to 20 to minimize training time.
# You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
# However, for better model performance, consider increasing the number of epochs to those specified in the referenced paper.
epochs = 20  
batch_size = 16  # Batch size
n_out = target_train.n_labels

for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))

    target_train_select, target_train_re = subsample(target_train, size)

    loader_source_tr = DisjointLoader(source_train, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_source_te = DisjointLoader(source_test,  batch_size=batch_size)


    loader_target_tr = DisjointLoader(target_train_select, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_target_te = DisjointLoader(target_test, batch_size=batch_size)

    # build model
    GIN = GIN0(channels, layers)


    model =DANN_GIN(loader_source_tr, loader_target_tr, loader_target_te, GIN, n_out, epochs==20) # change the epochs here as well


    G, C = model.train()


    ################################################################################
    # Evaluate model
    ################################################################################
    results = []
    step = 0

    while step < loader_target_te.steps_per_epoch:      
        step += 1
        inputs, target = loader_target_te.__next__()
        pred = C(G(inputs, training=False), training=False)
        results.append(
            (
               f1_score(np.argmax(target, axis=1), np.argmax(pred, axis=1), average='weighted')
            )
        )
    print("Done. Test f1: {}".format(np.mean(results, 0)))

        
       

--------------------------Sample size 20-------------------------
Epoch: 10
C_loss train: 0.1980, Acc train source: 88.75 , Acc train target: 93.00
D_loss train: 1.5630, G_loss train: 0.3710
Test results - Loss: 0.023 - Acc: 0.992
Epoch: 20
C_loss train: 0.1596, Acc train source: 92.50 , Acc train target: 95.75
D_loss train: 1.3979, G_loss train: 0.3319
Test results - Loss: 0.034 - Acc: 0.989
Done. Test f1: 0.9885647099562784
--------------------------Sample size 50-------------------------
Epoch: 10
C_loss train: 0.4550, Acc train source: 93.75 , Acc train target: 93.40
D_loss train: 1.5122, G_loss train: 0.6289
Test results - Loss: 0.012 - Acc: 0.997
Epoch: 20
C_loss train: 0.1730, Acc train source: 90.62 , Acc train target: 95.40
D_loss train: 1.3883, G_loss train: 0.3447
Test results - Loss: 0.019 - Acc: 0.998
Done. Test f1: 0.9975266640336484
--------------------------Sample size 100-------------------------
Epoch: 10
C_loss train: 0.1664, Acc train source: 91.52 , Acc train targe

## Set kelihos_v3 as the target domain

In [20]:
# Kelihos_ver3 label is 3
target_malware, source_malware = filter_label(malware, [2])


# convert it to binary labels
target_malware = binary_label(target_malware, True)
source_malware  = binary_label(source_malware, True)
target_normal = binary_label(target_normal, False)
source_normal = binary_label(source_normal, False)

print("Target malware size: {}".format(len(target_malware)))
print("Target normal size: {}".format(len(target_normal)))
print("Source malware size: {}".format(len(source_malware)))
print("Source normal size: {}".format(len(source_normal)))
           
           

Target malware size: 2919
Target normal size: 5768
Source malware size: 7792
Source normal size: 6510


In [21]:
# Split the  malware/benign dataset into train and test set
target_malware_train, target_malware_test = train_test_split(target_malware, 0.5)
print(len(target_malware_train))
print(len(target_malware_test))
# Split the normal dataset into train and test set
target_normal_train, target_normal_test = train_test_split(target_normal, 0.5)
print(len(target_normal_train))
print(len(target_normal_test))

1459
1460
2884
2884


In [22]:
# combine the normal and malware dataset
source = merge_dataset(source_normal, source_malware)

# source dataset train and test split
source_train, source_test = train_test_split(source, 0.75)
# we do the same for the target dataset
target_train = merge_dataset(target_normal_train, target_malware_train)
target_test = merge_dataset(target_normal_test, target_malware_test)



print("Target train dataset size: {}".format(len(target_train)))
print("Target test dataset size: {}".format(len(target_test)))
print("Source train dataset size: {}".format(len(source_train)))
print("Source test dataset size: {}".format((len(source_test))))

Target train dataset size: 4343
Target test dataset size: 4344
Source train dataset size: 10726
Source test dataset size: 3576


In [23]:
samples = [20, 50, 100, 200, 300, 500]

channels = 256  # Hidden units
layers = 3  # GIN layers
# We have limited the number of training epochs to 20 to minimize training time.
# You can change the epochs to a lower number (lowest 1) just to test if the code is functional.
# However, for better model performance, consider increasing the number of epochs to those specified in the referenced paper
epochs = 20  
batch_size = 16  # Batch size
n_out = target_train.n_labels

for size in samples: 
    print("--------------------------Sample size {}-------------------------".format(size))
    
    target_train_select, target_train_re = subsample(target_train, size)

    loader_source_tr = DisjointLoader(source_train, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_source_te = DisjointLoader(source_test,  batch_size=batch_size)


    loader_target_tr = DisjointLoader(target_train_select, batch_size=batch_size, epochs = epochs, shuffle = True)
    loader_target_te = DisjointLoader(target_test, batch_size=batch_size)

    # build model
    GIN = GIN0(channels, layers)


    model =DANN_GIN(loader_source_tr, loader_target_tr, loader_target_te, GIN, n_out, epochs==20) # change the epochs here as well


    G, C = model.train()


    ################################################################################
    # Evaluate model
    ################################################################################
    results = []
    step = 0

    while step < loader_target_te.steps_per_epoch:      
        step += 1
        inputs, target = loader_target_te.__next__()
        pred = C(G(inputs, training=False), training=False)
        results.append(
            (
               f1_score(np.argmax(target, axis=1), np.argmax(pred, axis=1), average='weighted')
            )
        )
    print("Done. Test f1: {}".format(np.mean(results, 0)))
        
        
       

--------------------------Sample size 20-------------------------
Epoch: 10
C_loss train: 0.2245, Acc train source: 87.50 , Acc train target: 94.00
D_loss train: 1.6076, G_loss train: 0.4152
Test results - Loss: 0.083 - Acc: 0.982
Epoch: 20
C_loss train: 0.2095, Acc train source: 94.38 , Acc train target: 95.75
D_loss train: 1.3143, G_loss train: 0.3941
Test results - Loss: 0.325 - Acc: 0.908
Done. Test f1: 0.8998785758463274
--------------------------Sample size 50-------------------------
Epoch: 10
C_loss train: 0.3295, Acc train source: 84.06 , Acc train target: 97.00
D_loss train: 1.4786, G_loss train: 0.5102
Test results - Loss: 0.270 - Acc: 0.845
Epoch: 20
C_loss train: 0.2838, Acc train source: 86.41 , Acc train target: 97.60
D_loss train: 1.2430, G_loss train: 0.4757
Test results - Loss: 0.128 - Acc: 0.975
Done. Test f1: 0.9750011629652922
--------------------------Sample size 100-------------------------
Epoch: 10
C_loss train: 0.0935, Acc train source: 91.52 , Acc train targe